# Cuaderno 2: Simulación del tsunami en TPU

Tsunami del 12 de diciembre de 1979 (Mw 8.2, Tumaco) — **tsunamiTPUlab v1.0**.

**⚠ REQUISITO:** Runtime **TPU** activo antes de ejecutar.
`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **TPU**

Luego: `Entorno de ejecución` → `Ejecutar todo`.

In [ ]:
# Celda 1/8 — Instalar dependencias
%pip install -q tensorflow scikit-image rasterio pyproj
print('✓ Dependencias listas')

In [ ]:
# Celda 2/8 — Imports y clonar modelo
import sys, os
import numpy as np
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling

IN_COLAB   = 'google.colab' in sys.modules
WORK_DIR   = Path('/content') if IN_COLAB else Path('.')
OUTPUT_DIR = WORK_DIR / 'output_1979'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEM_PATH   = WORK_DIR / 'tumaco_dem_utm_sim.tif'

if IN_COLAB:
    if not os.path.exists('/content/tsunamiTPUlab'):
        os.system('git clone --quiet https://github.com/smarras79/tsunamiTPUlab.git /content/tsunamiTPUlab')
    MODEL_DIR = Path('/content/tsunamiTPUlab')
    sys.path.insert(0, str(MODEL_DIR))
else:
    MODEL_DIR = Path('./tsunamiTPUlab')

print(f'✓ Modelo en: {MODEL_DIR}')

In [ ]:
# Celda 3/8 — Preparar DEM (auto-genera si no existe)
if not DEM_PATH.exists():
    print('DEM no encontrado — generando...')
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 0.5, 3.5, -79.5, -77.0
    CRS_SRC  = CRS.from_epsg(4326)
    CRS_DEST = CRS.from_epsg(32618)
    geo_path = WORK_DIR / 'tumaco_bathy_geo.tif'
    bathy_ok = False

    # Intento 1: GEBCO 2023
    try:
        url = (f'https://download.gebco.net/a/gebco_2023'
               f'?lat1={LAT_MIN}&lat2={LAT_MAX}&lon1={LON_MIN}&lon2={LON_MAX}&format=netcdf4')
        r = requests.get(url, timeout=120)
        if r.status_code == 200 and len(r.content) > 5000:
            import xarray as xr, io
            ds      = xr.open_dataset(io.BytesIO(r.content))
            varname = [v for v in ds.data_vars if 'elev' in v.lower() or 'z' in v.lower()][0]
            da      = ds[varname]
            lat_dim = [d for d in da.dims if 'lat' in d.lower()][0]
            lon_dim = [d for d in da.dims if 'lon' in d.lower()][0]
            if da[lat_dim].values[0] < da[lat_dim].values[-1]:
                da = da.isel({lat_dim: slice(None, None, -1)})
            arr = da.values.astype('float32')
            lats, lons = da[lat_dim].values, da[lon_dim].values
            tf_g = from_bounds(lons.min(), lats.min(), lons.max(), lats.max(),
                               arr.shape[1], arr.shape[0])
            with rasterio.open(geo_path, 'w', driver='GTiff',
                               height=arr.shape[0], width=arr.shape[1],
                               count=1, dtype='float32', crs=CRS_SRC, transform=tf_g) as dst:
                dst.write(arr, 1)
            bathy_ok = True
            print('  ✓ GEBCO 2023')
    except Exception as e:
        print(f'  ✗ GEBCO: {e}')

    # Intento 2: ETOPO 2022 (NOAA)
    if not bathy_ok:
        try:
            nc = int((LON_MAX - LON_MIN) / 0.004167)
            nr = int((LAT_MAX - LAT_MIN) / 0.004167)
            url2 = ('https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/'
                    'DEM_global_mosaic/ImageServer/exportImage'
                    f'?bbox={LON_MIN},{LAT_MIN},{LON_MAX},{LAT_MAX}'
                    f'&bboxSR=4326&size={nc},{nr}&imageSR=4326&format=tiff&pixelType=F32&f=image')
            r2 = requests.get(url2, timeout=120)
            r2.raise_for_status()
            with open(geo_path, 'wb') as f:
                f.write(r2.content)
            bathy_ok = True
            print('  ✓ ETOPO 2022')
        except Exception as e:
            print(f'  ✗ ETOPO: {e}')

    # Intento 3: DEM sintético
    if not bathy_ok:
        print('  ⚠ DEM sintético (solo pruebas)')
        res = 0.004167
        lons_s = np.arange(LON_MIN, LON_MAX, res)
        lats_s = np.arange(LAT_MIN, LAT_MAX, res)
        nr, nc = len(lats_s), len(lons_s)
        xr2 = (lons_s - LON_MIN) / (LON_MAX - LON_MIN)
        z = np.where(xr2 < 0.70, -4000 + 3800*(xr2/0.70),
            np.where(xr2 < 0.88, -200 + 200*((xr2-0.70)/0.18), 10.0))
        dem_s = np.tile(z, (nr, 1)).astype('float32')
        dem_s += gaussian_filter(np.random.randn(nr, nc)*50, sigma=5)
        tf_s  = from_bounds(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, nc, nr)
        with rasterio.open(geo_path, 'w', driver='GTiff', height=nr, width=nc,
                           count=1, dtype='float32', crs=CRS_SRC, transform=tf_s) as dst:
            dst.write(dem_s, 1)

    # Reproyectar a UTM 18N
    utm_path = WORK_DIR / 'tumaco_dem_utm.tif'
    with rasterio.open(geo_path) as src:
        tf_utm, w_utm, h_utm = calculate_default_transform(
            src.crs, CRS_DEST, src.width, src.height, *src.bounds)
        meta = src.meta.copy()
        meta.update({'crs': CRS_DEST, 'transform': tf_utm,
                     'width': w_utm, 'height': h_utm, 'dtype': 'float32'})
        with rasterio.open(utm_path, 'w', **meta) as dst:
            reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=tf_utm, dst_crs=CRS_DEST,
                      resampling=Resampling.bilinear)

    # Invertir eje X (océano a y=L)
    with rasterio.open(utm_path) as src:
        arr_u = src.read(1);  old_tf = src.transform;  meta2 = src.meta.copy()
    arr_flip = arr_u[:, ::-1]
    meta2['transform'] = rasterio.transform.from_origin(
        west=old_tf.c + old_tf.a*arr_u.shape[1], north=old_tf.f,
        xsize=-old_tf.a, ysize=-old_tf.e)
    with rasterio.open(DEM_PATH, 'w', **meta2) as dst:
        dst.write(arr_flip, 1)
    print(f'  ✓ DEM guardado: {DEM_PATH}')

with rasterio.open(str(DEM_PATH)) as src:
    _sh = (src.height, src.width);  _res = abs(src.transform.a)
print(f'✓ DEM: {_sh[0]}×{_sh[1]} px | {_res:.0f} m/px')

In [ ]:
# Celda 4/8 — Configurar TPU
PUBLIC_COLAB = IN_COLAB
TPU_WORKER   = ''

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
print(f'TensorFlow {tf.__version__}')

if PUBLIC_COLAB:
    tpu_addr = os.environ.get('COLAB_TPU_ADDR', '')
    if not tpu_addr:
        raise RuntimeError('TPU no detectada. Active runtime TPU y vuelva a ejecutar.')
    TPU_WORKER = 'grpc://' + tpu_addr
    print(f'✓ TPU: {TPU_WORKER}')

In [ ]:
# Celda 5/8 — Cargar tsunamiTPUlab
os.chdir(str(MODEL_DIR))
with open(MODEL_DIR / 'user_constants.py', 'w') as f:
    f.write(f"BUCKET=''\nPROJECT_ID=''\nPUBLIC_COLAB={PUBLIC_COLAB}\nTPU_WORKER='{TPU_WORKER}'\n")

import tpu_simulation_utilities as tsu
print('✓ tsunamiTPUlab cargado')

In [ ]:
# Celda 6/8 — Parámetros + N-wave 1979
with rasterio.open(str(DEM_PATH)) as src:
    RESOLUTION = int(abs(src.transform.a))
    DEM_SHAPE  = (src.height, src.width)

DT, CX, CY          = 1.0, 1, 8
NUM_SECS             = 1800.0
SECS_PER_CYCLE       = 60.0
RUN_DIR              = str(OUTPUT_DIR)
print(f'Resolución: {RESOLUTION}m | Shape: {DEM_SHAPE} | {NUM_SECS/60:.0f} min')

def nwave_tumaco_1979(y, x, Ly, nx, ny, dem):
    A, lamb = 1.5, 80_000.0
    k1, k2  = 28.416/lamb**2, 256.0/lamb**2
    y1 = (Ly - 60_000.0) + 0.3153*lamb
    y2 =  Ly - 60_000.0
    z  = 2.0*(A*tf.math.exp(-k1*tf.math.square(y-y1))
              - (A/3.0)*tf.math.exp(-k2*tf.math.square(y-y2)))
    z  = tf.expand_dims(z, 0)
    z  = tf.repeat(z, nx, axis=0)
    z  = z - tf.math.minimum(dem, tf.zeros_like(dem))
    z  = tf.where(z < tsu.SAINT_VENANT_EPS, tf.ones_like(z)*tsu.SAINT_VENANT_EPS, z)
    return z

print('✓ N-wave definida')

In [ ]:
# Celda 7/8 — Ejecutar simulación en TPU (~5-10 min)
import time

def build_and_run():
    slope = 1e-4
    dem_arr, _ = tsu.load_dem_from_tiff_file(str(DEM_PATH), RESOLUTION, smooth_kernel=5)
    print(f'DEM: {dem_arr.shape} | min={dem_arr.min():.0f}m max={dem_arr.max():.0f}m')
    mask    = np.ones_like(dem_arr, dtype=bool)
    manning = tsu.get_manning_matrix_from_river_mask(mask, tsu.MANNING_COEFF_RIVER, tsu.MANNING_COEFF_FLOODPLAIN)
    params  = tsu.get_sv_params(dem_arr.shape, RESOLUTION, NUM_SECS, SECS_PER_CYCLE, CX, CY, DT)
    print(f'Grid: nx={params.nx} ny={params.ny} steps={params.num_steps}')

    bc = dict(fraction_start=0, fraction_end=1.0,
              left_padding=params.left_padding, top_padding=params.top_padding,
              slope=slope, value=0.0, unpadded_dem=dem_arr, unpadded_manning_matrix=manning)
    h_bcs  = [tsu.NeumannBoundary(tsu.BoundarySide.LEFT, **bc),
               tsu.NeumannBoundary(tsu.BoundarySide.RIGHT, **bc)]
    qy_bcs = [tsu.NeumannBoundary(tsu.BoundarySide.LEFT, **bc),
               tsu.DirichletBoundary(tsu.BoundarySide.RIGHT, **bc)]
    qx_bcs = [tsu.NeumannBoundary(tsu.BoundarySide.TOP, **bc),
               tsu.NeumannBoundary(tsu.BoundarySide.BOTTOM, **bc)]

    ifm     = tsu.InitFilesManager(params, tsu.three_d_subgrid_of_2d_grid, RUN_DIR)
    init_fn = tsu.SaintVenantRealisticInitFnBuilder(
        params, tsu.INIT_STATE_KEYS, dem_arr, manning, ifm,
        None, 0.0, nwave_tumaco_1979, tsu.constant_height_zeros)
    step_fn = tsu.SaintVenantRiverChannelStep(tsu.ApplyKernelOp(), params, h_bcs, qx_bcs, qy_bcs)
    step_fn = tsu.DynamicStepUpdater(step_fn, num_secs_per_while_loop=SECS_PER_CYCLE, dt=DT)
    step_fn = tsu.build_step_fn(params, step_fn, tsu.STATE_KEYS, tsu.ADDITIONAL_STATE_KEYS)

    def sim_builder(sess):
        topo = tf.tpu.experimental.Topology(sess.run(tf.tpu.initialize_system()))
        return tsu.TPUSimulation(init_fn=init_fn.init_fn, step_fn=step_fn,
                                 computation_shape=params.computation_shape,
                                 tpu_topology=topo, host_vars_filter=tsu.INIT_STATE_KEYS)

    config = tf.ConfigProto(isolate_session_state=True)
    sess   = tf.Session(config=config, target=TPU_WORKER)
    sim    = sim_builder(sess)
    runner = tsu.get_session_runner_builder(params, ifm, RUN_DIR, sim)
    sess.run(tf.global_variables_initializer())
    runner(sess, 0)
    try: tf.tpu.shutdown_system()
    except: pass
    print('✓ Simulación completada')

if PUBLIC_COLAB:
    t0 = time.time()
    with tf.Graph().as_default():
        build_and_run()
    print(f'Tiempo total: {(time.time()-t0)/60:.1f} min')
else:
    print('Modo local — sin ejecución TPU')

In [ ]:
# Celda 8/8 — Verificar salida y vista previa
import glob

output_files = sorted(glob.glob(f'{RUN_DIR}/h-*.np'))
if output_files:
    total_mb = sum(Path(f).stat().st_size for f in output_files) / 1e6
    print(f'✓ {len(output_files)} snapshots | {total_mb:.1f} MB')
    print(f'  {Path(output_files[0]).name} → {Path(output_files[-1]).name}')

    with rasterio.open(str(DEM_PATH)) as src:
        dem_sim = src.read(1)
    t_files = {float(Path(f).stem.split('-')[1]): f for f in output_files}
    t_close = min(t_files, key=lambda t: abs(t - 900.0))
    hmap    = np.load(t_files[t_close])
    inund   = np.where(dem_sim >= 0, hmap + dem_sim, 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    im1 = axes[0].imshow(hmap,  cmap='RdBu_r', vmin=-3, vmax=3)
    plt.colorbar(im1, ax=axes[0], label='Η (m)')
    axes[0].set_title(f't = {t_close:.0f}s ({t_close/60:.0f} min)')
    im2 = axes[1].imshow(inund, cmap='YlOrRd',  vmin=0,  vmax=5)
    plt.colorbar(im2, ax=axes[1], label='Inundación (m)')
    axes[1].set_title('Inundación sobre terreno')
    plt.suptitle('Tsunami Tumaco 1979 — Vista previa')
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'preview_t900s.png', dpi=150)
    plt.show()
    print(f'Inundación máx.: {inund.max():.2f} m')
else:
    print('⚠ Sin archivos de salida en:', RUN_DIR)